In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Administrator\Desktop\data_learn\Hot-Drink-Retail-Analysis\data\processed\result_pre_sale.csv")

def get_weather_type(temp):
    if temp < -5:
        return '1_极寒挑战期'
    elif -5 <= temp < 5:
        return '2_降温敏感期'
    else:
        return '3_常温成长期'

df['weather_type'] = df['tmin'].apply(get_weather_type)

df_report_loc_temp = df.groupby(['location_type','weather_type'])['pre_sale'].mean().unstack().round(2)

print("--- 门店选址抗寒韧性分析矩阵 ---")
display(df_report_loc_temp)

df_report_loc_temp['抗寒韧性(%)'] = (df_report_loc_temp['1_极寒挑战期'] / df_report_loc_temp['3_常温成长期'] * 100.0).round(2)
display(df_report_loc_temp[['抗寒韧性(%)']])

--- 门店选址抗寒韧性分析矩阵 ---


weather_type,1_极寒挑战期,2_降温敏感期,3_常温成长期
location_type,,,
中庸：混合社区,927.86,1130.70,1269.67
抗寒：重度商务刚需,1079.56,1149.52,1269.63
脆弱：极度依赖天气,625.50,1092.98,1269.63


weather_type,抗寒韧性(%)
location_type,
中庸：混合社区,73.08
抗寒：重度商务刚需,85.03
脆弱：极度依赖天气,49.27


In [13]:
import psycopg2

conn = psycopg2.connect(
    host = 'localhost',
    port = '5432',
    database = 'postgres',
    user = 'postgres',
    password = ''
)

cur = conn.cursor()

query = """
DROP TABLE IF EXISTS hot_drink_retail.report_loc_temp;

CREATE TABLE hot_drink_retail.report_loc_temp AS
SELECT 
    location_type,
    ROUND(AVG(CASE WHEN tmin < -5 THEN pre_sale END), 2) AS "1_极寒挑战期",
    ROUND(AVG(CASE WHEN tmin >= -5 AND tmin < 5 THEN pre_sale END), 2) AS "2_降温敏感期",
    ROUND(AVG(CASE WHEN tmin >= 5 THEN pre_sale END), 2) AS "3_常温成长期",

    ROUND(
        AVG(CASE WHEN tmin < -5 THEN pre_sale END) * 100.0 /
        NULLIF(AVG(CASE WHEN tmin >= 5 THEN pre_sale END), 0), 2
        ) AS "抗寒韧性_百分比"
FROM hot_drink_retail.result_pre_sale
GROUP BY location_type;
"""

cur.execute(query)
conn.commit()

df_sql_loc_temp = pd.read_sql("SELECT * FROM hot_drink_retail.report_loc_temp", conn)

cur.close()
conn.close()
print("--- SQL 门店选址抗寒韧性分析矩阵 ---")
display(df_sql_loc_temp.head())

df_sql_loc_temp.to_csv(r"C:\Users\Administrator\Desktop\data_learn\Hot-Drink-Retail-Analysis\data\processed\04_report_loc_temp.csv", index=False, encoding='utf-8-sig')

--- SQL 门店选址抗寒韧性分析矩阵 ---


C:\Users\Administrator\AppData\Local\Temp\ipykernel_9392\3436811282.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql_loc_temp = pd.read_sql("SELECT * FROM hot_drink_retail.report_loc_temp", conn)


,location_type,1_极寒挑战期,2_降温敏感期,3_常温成长期,抗寒韧性_百分比
0,中庸：混合社区,927.86,1130.70,1269.67,73.08
1,抗寒：重度商务刚需,1079.56,1149.52,1269.63,85.03
2,脆弱：极度依赖天气,625.50,1092.98,1269.63,49.27


In [2]:
df['brand_group'] = '新锐/独立品牌'

df.loc[df['name'].str.contains('Luckin|瑞幸', case=False, na=False), 'brand_group'] = '瑞幸咖啡' #.str.contains()会返回布尔值
df.loc[df['name'].str.contains('Starbucks|星巴克', case=False, na=False), 'brand_group'] = '星巴克'
df.loc[df['name'].str.contains('喜茶', case=False, na=False), 'brand_group'] = '喜茶'
df.loc[df['name'].str.contains('奈雪', case=False, na=False), 'brand_group'] = '奈雪'
df.loc[df['name'].str.contains('霸王茶姬', case=False, na=False), 'brand_group'] = '霸王茶姬'
df.loc[df['name'].str.contains('茶百道', case=False, na=False), 'brand_group'] = '茶百道'
df.loc[df['name'].str.contains('蜜雪', case=False, na=False), 'brand_group'] = '蜜雪'


brand_report = df.groupby(['brand_group', 'location_type', 'weather_type'])['pre_sale'].mean().unstack().round(2)
brand_report['韧性%'] = (brand_report['1_极寒挑战期'] / brand_report['3_常温成长期'] * 100).round(1)

print("--- 品牌与气候韧性矩阵 ---")
#display(brand_report.sort_values('韧性%', ascending=False))
display(brand_report)

--- 品牌与气候韧性矩阵 ---


weather_type               1_极寒挑战期  2_降温敏感期  3_常温成长期   韧性%
brand_group location_type                                 
喜茶          中庸：混合社区         927.71  1130.89  1269.75  73.1
            抗寒：重度商务刚需      1079.39  1149.43  1269.70  85.0
奈雪          中庸：混合社区         927.49  1131.04  1269.80  73.0
            抗寒：重度商务刚需      1079.79  1150.17  1269.91  85.0
            脆弱：极度依赖天气       626.06  1092.89  1269.44  49.3
新锐/独立品牌     中庸：混合社区         927.86  1130.69  1269.65  73.1
            抗寒：重度商务刚需      1079.51  1149.53  1269.66  85.0
            脆弱：极度依赖天气       625.45  1092.76  1269.66  49.3
星巴克         中庸：混合社区         927.87  1130.73  1269.69  73.1
            抗寒：重度商务刚需      1079.76  1149.57  1269.59  85.0
            脆弱：极度依赖天气       625.75  1093.19  1270.22  49.3
瑞幸咖啡        中庸：混合社区         927.77  1130.70  1269.67  73.1
            抗寒：重度商务刚需      1079.58  1149.46  1269.51  85.0
            脆弱：极度依赖天气       624.57  1093.73  1269.15  49.2
茶百道         中庸：混合社区         928.13  1130.35  1269.83  73.1
            抗寒：重度商务刚需      1080.04  1149.41  1269.45  85.1
            脆弱：极度依赖天气       625.81  1092.64  1269.37  49.3
蜜雪          中庸：混合社区         928.14  1130.64  1269.65  73.1
            抗寒：重度商务刚需      1079.76  1149.49  1269.83  85.0
霸王茶姬        中庸：混合社区         927.92  1130.54  1269.73  73.1
            抗寒：重度商务刚需      1079.19  1149.50  1269.53  85.0
            脆弱：极度依赖天气       625.63  1093.84  1269.76  49.3